In [1]:
# Now load autoreload properly
%load_ext autoreload
%autoreload 2
import os
import string
import pandas as pd
import numpy as np
import datetime
# set printing all columns
pd.set_option('display.max_columns', None)
from DataEncoder import deterministic_length_stratified_split
from pathlib import Path

Loading all sets except sepsis data 

In [8]:
dataname = "helpdesk"
#dataname = "BPI12"
#dataname = "BPI12W"
#dataname = "BPI13I"
#dataname = "BPI13C"

if os.name == "nt":   # Windows
    file = "D:/Research/nextevent/data/" + dataname +".csv"
else:                 # Linux / server
    file = "~/Research_in_UAE/nextevent/data/"+ dataname +".csv"

data_raw = pd.read_csv(file)

Prepare Sepsis

In [2]:
dataname = "sepsis"

data_raw = pd.read_csv("D:/Research/nextevent/data/Sepsis Cases - Event Log.csv")

data_raw = data_raw.dropna(subset=['case:concept:name']).reset_index(drop=True)
data_raw['case:concept:name'].isna().sum()

data_raw.sort_values(by=['case:concept:name', 'time:timestamp'], inplace=True)

grp_col = 'case:concept:name'

# pick all columns that appear *before* the group identifier in your dataframe
cols_before = list(data_raw.columns[:data_raw.columns.get_loc(grp_col)])

# forward-fill then back-fill *within each case*, then fill groups that had no value at all with 'na'
data_raw[cols_before] = (
    data_raw
    .groupby(grp_col, sort=False)[cols_before]
    .transform(lambda s: s.ffill().bfill())
    .fillna('missing')   # if an entire group/column was NaN, set to 'na'
)

# pick all columns that appear *before* the group identifier in your dataframe
cols_after = list(data_raw.columns[data_raw.columns.get_loc(grp_col)+1:])

data_raw[cols_after] = (
    data_raw
      .groupby(grp_col, sort=False)[cols_after]
      .ffill()
      .fillna(0)
)

data_raw = data_raw.replace("?", "missing")

BPI20

In [30]:
dataname = "BPI20"

data_raw = pd.read_csv("D:/Research/nextevent/data/BPI20PrepaidTravelCost.csv")

data_raw = data_raw.dropna(subset=['case:concept:name']).reset_index(drop=True)
data_raw['case:concept:name'].isna().sum()

data_raw = data_raw[['org:role', 'concept:name', 'time:timestamp', 'case:OrganizationalEntity', 'case:RequestedAmount', 'case:Project', 'case:concept:name', "case:Permit RequestedBudget"]]

# First change NAN to preseve the orginal NAN as categorial in org:rescource
data_raw = data_raw.fillna('missing')

In [5]:
dataname = "BPI20R"

data_raw = pd.read_csv("D:/Research/nextevent/data/BPI20RequestForPayment.csv")

data_raw = data_raw.dropna(subset=['case:concept:name']).reset_index(drop=True)
data_raw['case:concept:name'].isna().sum()

data_raw = data_raw[['org:resource', 'org:role', 'concept:name', 'time:timestamp', 'case:OrganizationalEntity', 'case:RequestedAmount', 'case:RfpNumber', 'case:Task', 'case:Project', 'case:Activity', 'case:concept:name']]

#First change NAN to preseve the orginal NAN as categorial in org:rescource
data_raw = data_raw.fillna('missing')

Prepare bpi17

In [ ]:
dataname = "BPI17"

data_raw = pd.read_csv("D:/Research/nextevent/data/BPI Challenge 2017.csv")

data_raw = data_raw.dropna(subset=['case:concept:name']).reset_index(drop=True)
data_raw['case:concept:name'].isna().sum()

data_raw.sort_values(by=['case:concept:name', 'time:timestamp'], inplace=True)


grp_col = 'case:concept:name'

# pick all columns that appear *before* the group identifier in your dataframe
#cols_before = list(data_raw.columns[:data_raw.columns.get_loc(grp_col)])
col_num = ["FirstWithdrawalAmount", "NumberOfTerms" , "MonthlyCost", "CreditScore",	"OfferedAmount"]
col_bol = ["Accepted", "Selected"]
col_cat = ["OfferID"]

# back-fill *within each case*, then fill groups that had no value at all with 'na'
data_raw[col_num] = (
    data_raw
    .groupby(grp_col, sort=False)[col_num]
    .ffill()
    .fillna(0)   # if an entire group/column was NaN, set to 'na'
)


data_raw[col_bol] = (
    data_raw
      .groupby(grp_col, sort=False)[col_bol]
      .ffill()
      .fillna(False)
)

data_raw[col_cat] = (
    data_raw
      .groupby(grp_col, sort=False)[col_cat]
      .ffill()
      .fillna("missing")
)



data_raw = data_raw.replace("?", "missing")
data_raw = data_raw.fillna('missing')

Prepossing All sets except sepsis and BPI17

In [7]:
# First change NAN to preseve the orginal NAN as categorial in org:rescource
data_raw = data_raw.fillna('missing')

In [8]:
data_raw.sort_values(by=['case:concept:name', 'time:timestamp'], inplace=True)

Now for all sets

In [9]:
data_raw = data_raw.drop_duplicates().reset_index(drop=True)

In [10]:
data_raw['time:timestamp'] = pd.to_datetime(data_raw['time:timestamp'], format='ISO8601').dt.strftime('%Y-%m-%d %H:%M:%S')

Bpi 12 and bpi 13, 17 combine the concept:name + lifecycle:transition, no need for others

In [27]:
data_raw = data_raw.rename(columns={"concept:name": "concept:name_a"})
data_raw["concept:name"] = data_raw["concept:name_a"].astype(str) + "_" + data_raw["lifecycle:transition"].astype(str)

In [13]:
# helpdesk: drop invalid columns
data_raw = data_raw.drop(columns = ['lifecycle:transition', 'Activity', 'Resource', 'case:variant-index', 'case:creator'])
# BPI12
#data_raw = data_raw.drop(columns = ['lifecycle:transition', 'concept:name_a', "case:REG_DATE"])
# BPI13: drop invalid columns
#data_raw = data_raw.drop(columns = ['lifecycle:transition', 'concept:name_a'])
#sepis
#data_raw = data_raw.drop(columns = ['lifecycle:transition'])
#BPI17: drop invalid columns
#data_raw = data_raw.drop(columns = ['lifecycle:transition', 'concept:name_a', "EventID"])
# BPI20: no drop

In [11]:
data_raw.to_csv("../output/data_processed/"+ dataname + "_full.csv", index = False)

In [12]:
# Use stratified sampling over sequence length to preserve distributional characteristics
train_indices, test_indices = deterministic_length_stratified_split(data_raw, "case:concept:name", "concept:name", test_size=0.1, n_bins=10, random_state=42, verbose=True)

train_event = data_raw[data_raw["case:concept:name"].isin(train_indices)]
test_event = data_raw[data_raw["case:concept:name"].isin(test_indices)]

case_lengths = data_raw.groupby("case:concept:name").size()
train_lengths = case_lengths.loc[train_indices]
test_lengths  = case_lengths.loc[test_indices]

print(f"Train set: {len(train_indices)} samples")
print(f"Train length range: {min(train_lengths)} - {max(train_lengths)}")
print(f"Train length mean: {np.mean(train_lengths):.2f}")

print(f"\nTest set: {len(test_indices)} samples") 
print(f"Test length range: {min(test_lengths)} - {max(test_lengths)}")
print(f"Test length mean: {np.mean(test_lengths):.2f}")

Train cases: 6198 | Test cases: 688
Train set: 6198 samples
Train length range: 1 - 20
Train length mean: 5.33

Test set: 688 samples
Test length range: 3 - 14
Test length mean: 5.50


In [13]:
train_event.to_csv("../output/data_processed/" + dataname + "_train.csv", index = False)
test_event.to_csv("../output/data_processed/"+ dataname + "_hold.csv", index = False)

In [48]:
dataname="BPI17"
data = pd.read_csv("../output/data_processed/"+ dataname + "_hold.csv")

In [49]:
data.columns

Index(['Action', 'org:resource', 'EventOrigin', 'time:timestamp',
       'case:LoanGoal', 'case:ApplicationType', 'case:concept:name',
       'case:RequestedAmount', 'FirstWithdrawalAmount', 'NumberOfTerms',
       'Accepted', 'MonthlyCost', 'Selected', 'CreditScore', 'OfferedAmount',
       'OfferID', 'concept:name'],
      dtype='str')

In [ ]:
if dataname == "helpdesk":
    cat_cols_event = ['org:resource']
    num_cols_event = []
    cat_cols_seq = ['case:variant']
    num_cols_seq = []
elif dataname == "BPI12" or dataname == "BPI12W":
    cat_cols_event = ['org:resource']
    num_cols_event = []
    cat_cols_seq = ['case:variant']
    num_cols_seq = []